# Model Performance Evaluation
Membuktikan Table IV dari paper: **"Cross-Validation Performance Comparison"**

Metrik: Accuracy, Precision, Recall, F1-Score, ROC-AUC, PR-AUC, FPR, FNR
Plus: Threshold-based baseline dan paired t-test

In [1]:
import pandas as pd
import numpy as np
import tempfile
import os
from scipy import stats
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')
print('Ready.')

Ready.


## 1. Load & Prepare Data

In [2]:
# --- Dataset A ---
df_a = pd.read_csv('SSH-Bruteforce.csv')
df_a.dropna(inplace=True)
numeric_cols = df_a.select_dtypes(include=[np.number]).columns
df_a = df_a[~np.isinf(df_a[numeric_cols]).any(axis=1)]
label_col = [c for c in df_a.columns if 'label' in c.lower()][0]
df_a[label_col] = df_a[label_col].apply(lambda x: 0 if 'benign' in str(x).lower() else 1)
attack_a = df_a[df_a[label_col] == 1]
benign_a = df_a[df_a[label_col] == 0].sample(n=len(attack_a), random_state=42)
df_a = pd.concat([benign_a, attack_a]).reset_index(drop=True)
X_a = df_a.drop(columns=[label_col]).select_dtypes(include=[np.number])
y_a = df_a[label_col]

# --- Dataset B ---
df_b = pd.read_csv('../Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv')
df_b.dropna(inplace=True)
if 'Timestamp' in df_b.columns:
    df_b = df_b.drop(columns=['Timestamp'])
numeric_cols_b = df_b.select_dtypes(include=[np.number]).columns
df_b = df_b[~np.isinf(df_b[numeric_cols_b]).any(axis=1)]
label_col_b = [c for c in df_b.columns if 'label' in c.lower()][0]
df_b[label_col_b] = df_b[label_col_b].apply(lambda x: 1 if 'ssh' in str(x).lower() else 0)
attack_b = df_b[df_b[label_col_b] == 1]
benign_b = df_b[df_b[label_col_b] == 0].sample(n=len(attack_b), random_state=42)
df_b = pd.concat([benign_b, attack_b]).reset_index(drop=True)
X_b = df_b.drop(columns=[label_col_b]).select_dtypes(include=[np.number])
y_b = df_b[label_col_b]

print(f'Dataset A: {X_a.shape[0]} samples, {X_a.shape[1]} features')
print(f'Dataset B: {X_b.shape[0]} samples, {X_b.shape[1]} features')

Dataset A: 375178 samples, 78 features
Dataset B: 375178 samples, 78 features


## 2. Get Top-10 Features

In [3]:
def get_top10(X, y):
    model = XGBClassifier(max_depth=6, n_estimators=100, learning_rate=0.3,
                          use_label_encoder=False, eval_metric='logloss', random_state=42)
    model.fit(X, y)
    booster = model.get_booster()
    importance = booster.get_score(importance_type='gain')
    feature_names = X.columns.tolist()
    imp_df = pd.DataFrame({
        'Feature': [feature_names[int(k.replace('f',''))] if k.startswith('f') and k[1:].isdigit() else k
                    for k in importance.keys()],
        'Gain': list(importance.values())
    }).sort_values('Gain', ascending=False)
    return imp_df.head(10)['Feature'].tolist()

top10_a = get_top10(X_a, y_a)
top10_b = get_top10(X_b, y_b)
print(f'Top-10 A: {top10_a}')
print(f'Top-10 B: {top10_b}')

Top-10 A: ['Dst Port', 'Init Bwd Win Byts', 'Fwd Seg Size Min', 'Flow Pkts/s', 'Bwd Header Len', 'Flow Duration', 'Fwd Pkt Len Max', 'Fwd Header Len', 'Init Fwd Win Byts', 'Bwd Pkt Len Mean']
Top-10 B: ['Init Bwd Win Byts', 'Dst Port', 'Fwd Seg Size Min', 'Init Fwd Win Byts', 'Flow IAT Std', 'Fwd IAT Min', 'Bwd Header Len', 'Bwd IAT Min', 'Flow IAT Max', 'Flow IAT Mean']


## 3. 5-Fold Cross-Validation (Full Metrics)

In [4]:
def evaluate_cv(X, y, top10_features, dataset_name):
    """
    5-Fold CV evaluation for 4 methods:
    Threshold-based, XGBoost, Random Forest, SVM
    Returns per-fold metrics for statistical tests.
    """
    X_opt = X[top10_features]
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    methods = {
        'Threshold-based': None,  # simple rule
        'XGBoost': XGBClassifier(max_depth=6, n_estimators=100, learning_rate=0.3,
                                 use_label_encoder=False, eval_metric='logloss', random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    }
    
    all_results = {}
    
    for method_name, model in methods.items():
        fold_metrics = {'acc': [], 'prec': [], 'rec': [], 'f1': [],
                        'roc_auc': [], 'pr_auc': [], 'fpr': [], 'fnr': []}
        
        for train_idx, test_idx in cv.split(X_opt, y):
            X_train, X_test = X_opt.iloc[train_idx], X_opt.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            
            if method_name == 'Threshold-based':
                # Simple rule: if Dst Port == 22 AND any flow metric > median -> attack
                # This simulates a naive threshold detector
                if 'Dst Port' in X_train.columns:
                    median_val = X_train.iloc[:, 0].median()
                    y_pred = (X_test.iloc[:, 0] > median_val).astype(int).values
                    y_prob = X_test.iloc[:, 0].values / X_test.iloc[:, 0].max()
                else:
                    median_val = X_train.iloc[:, 0].median()
                    y_pred = (X_test.iloc[:, 0] > median_val).astype(int).values
                    y_prob = X_test.iloc[:, 0].values / max(X_test.iloc[:, 0].max(), 1)
            elif method_name == 'SVM (RBF)':
                scaler = StandardScaler()
                X_tr_s = scaler.fit_transform(X_train)
                X_te_s = scaler.transform(X_test)
                model.fit(X_tr_s, y_train)
                y_pred = model.predict(X_te_s)
                y_prob = model.predict_proba(X_te_s)[:, 1]
            else:
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                if hasattr(model, 'predict_proba'):
                    y_prob = model.predict_proba(X_test)[:, 1]
                else:
                    y_prob = y_pred.astype(float)
            
            # Calculate metrics
            fold_metrics['acc'].append(accuracy_score(y_test, y_pred) * 100)
            fold_metrics['prec'].append(precision_score(y_test, y_pred, zero_division=0) * 100)
            fold_metrics['rec'].append(recall_score(y_test, y_pred, zero_division=0) * 100)
            fold_metrics['f1'].append(f1_score(y_test, y_pred, zero_division=0) * 100)
            
            try:
                fold_metrics['roc_auc'].append(roc_auc_score(y_test, y_prob))
                fold_metrics['pr_auc'].append(average_precision_score(y_test, y_prob))
            except:
                fold_metrics['roc_auc'].append(0.5)
                fold_metrics['pr_auc'].append(0.5)
            
            # FPR and FNR
            tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
            fold_metrics['fpr'].append(fp / (fp + tn) * 100 if (fp + tn) > 0 else 0)
            fold_metrics['fnr'].append(fn / (fn + tp) * 100 if (fn + tp) > 0 else 0)
        
        all_results[method_name] = fold_metrics
        print(f'  {method_name:18s} | Acc={np.mean(fold_metrics["acc"]):.2f}% | '
              f'Prec={np.mean(fold_metrics["prec"]):.2f}% | '
              f'Rec={np.mean(fold_metrics["rec"]):.2f}% | '
              f'F1={np.mean(fold_metrics["f1"]):.2f}%')
    
    return all_results

print('Function ready.')

Function ready.


## 4. Evaluation — Dataset A

In [5]:
print('='*75)
print('  MODEL PERFORMANCE - Dataset A (SSH vs Benign, Top-10 features)')
print('='*75)
results_a = evaluate_cv(X_a, y_a, top10_a, 'Dataset A')

  MODEL PERFORMANCE - Dataset A (SSH vs Benign, Top-10 features)
  Threshold-based    | Acc=0.81% | Prec=0.00% | Rec=0.00% | F1=0.00%
  XGBoost            | Acc=100.00% | Prec=100.00% | Rec=100.00% | F1=100.00%
  Random Forest      | Acc=100.00% | Prec=100.00% | Rec=100.00% | F1=100.00%
  SVM (RBF)          | Acc=99.98% | Prec=99.97% | Rec=100.00% | F1=99.98%


## 5. Evaluation — Dataset B

In [6]:
print('='*75)
print('  MODEL PERFORMANCE - Dataset B (SSH vs Non-SSH, Top-10 features)')
print('='*75)
results_b = evaluate_cv(X_b, y_b, top10_b, 'Dataset B')

  MODEL PERFORMANCE - Dataset B (SSH vs Non-SSH, Top-10 features)
  Threshold-based    | Acc=38.74% | Prec=0.11% | Rec=0.03% | F1=0.04%
  XGBoost            | Acc=99.99% | Prec=100.00% | Rec=99.98% | F1=99.99%
  Random Forest      | Acc=99.99% | Prec=100.00% | Rec=99.98% | F1=99.99%
  SVM (RBF)          | Acc=99.99% | Prec=99.99% | Rec=99.98% | F1=99.99%


## 6. Table IV Comparison

In [7]:
def print_table(results, dataset_name):
    print(f'\n{"="*80}')
    print(f'  TABLE IV ({dataset_name}) - Cross-Validation Performance (Mean +/- SD)')
    print(f'{"="*80}')
    print(f'{"Method":<18} | {"Accuracy (%)":<14} | {"Precision (%)":<14} | {"Recall (%)":<14} | {"F1-Score (%)":<14}')
    print(f'{"-"*80}')
    for method, metrics in results.items():
        acc = f'{np.mean(metrics["acc"]):.2f} +/- {np.std(metrics["acc"]):.2f}'
        prec = f'{np.mean(metrics["prec"]):.2f} +/- {np.std(metrics["prec"]):.2f}'
        rec = f'{np.mean(metrics["rec"]):.2f} +/- {np.std(metrics["rec"]):.2f}'
        f1 = f'{np.mean(metrics["f1"]):.2f} +/- {np.std(metrics["f1"]):.2f}'
        print(f'{method:<18} | {acc:<14} | {prec:<14} | {rec:<14} | {f1:<14}')
    print(f'{"="*80}')

print_table(results_a, 'Dataset A')
print_table(results_b, 'Dataset B')

print('\nPaper results (Table IV):')
print('  Threshold:  87.50 +/- 1.20 | 85.30 +/- 1.50 | 89.70 +/- 1.10 | 87.44 +/- 1.30')
print('  XGBoost:    99.85 +/- 0.02 | 99.82 +/- 0.03 | 99.88 +/- 0.02 | 99.85 +/- 0.02')
print('  R. Forest:  99.70 +/- 0.05 | 99.65 +/- 0.06 | 99.75 +/- 0.04 | 99.70 +/- 0.05')
print('  SVM:        94.20 +/- 0.40 | 93.50 +/- 0.45 | 94.80 +/- 0.35 | 94.14 +/- 0.40')


  TABLE IV (Dataset A) - Cross-Validation Performance (Mean +/- SD)
Method             | Accuracy (%)   | Precision (%)  | Recall (%)     | F1-Score (%)  
--------------------------------------------------------------------------------
Threshold-based    | 0.81 +/- 0.04  | 0.00 +/- 0.00  | 0.00 +/- 0.00  | 0.00 +/- 0.00 
XGBoost            | 100.00 +/- 0.00 | 100.00 +/- 0.00 | 100.00 +/- 0.00 | 100.00 +/- 0.00
Random Forest      | 100.00 +/- 0.00 | 100.00 +/- 0.00 | 100.00 +/- 0.00 | 100.00 +/- 0.00
SVM (RBF)          | 99.98 +/- 0.00 | 99.97 +/- 0.01 | 100.00 +/- 0.00 | 99.98 +/- 0.00

  TABLE IV (Dataset B) - Cross-Validation Performance (Mean +/- SD)
Method             | Accuracy (%)   | Precision (%)  | Recall (%)     | F1-Score (%)  
--------------------------------------------------------------------------------
Threshold-based    | 38.74 +/- 0.05 | 0.11 +/- 0.06  | 0.03 +/- 0.01  | 0.04 +/- 0.02 
XGBoost            | 99.99 +/- 0.00 | 100.00 +/- 0.00 | 99.98 +/- 0.01 | 99.99 +/-

## 7. Additional Metrics: ROC-AUC, PR-AUC, FPR, FNR

In [8]:
def print_additional_metrics(results, dataset_name):
    print(f'\n{"="*75}')
    print(f'  ADDITIONAL METRICS ({dataset_name})')
    print(f'{"="*75}')
    print(f'{"Method":<18} | {"ROC-AUC":<10} | {"PR-AUC":<10} | {"FPR (%)":<10} | {"FNR (%)":<10}')
    print(f'{"-"*75}')
    for method, metrics in results.items():
        roc = f'{np.mean(metrics["roc_auc"]):.4f}'
        pr = f'{np.mean(metrics["pr_auc"]):.4f}'
        fpr = f'{np.mean(metrics["fpr"]):.2f}'
        fnr = f'{np.mean(metrics["fnr"]):.2f}'
        print(f'{method:<18} | {roc:<10} | {pr:<10} | {fpr:<10} | {fnr:<10}')
    print(f'{"="*75}')
    print('Paper: XGBoost ROC-AUC=0.999, PR-AUC=0.998')

print_additional_metrics(results_a, 'Dataset A')
print_additional_metrics(results_b, 'Dataset B')


  ADDITIONAL METRICS (Dataset A)
Method             | ROC-AUC    | PR-AUC     | FPR (%)    | FNR (%)   
---------------------------------------------------------------------------
Threshold-based    | 0.0157     | 0.5038     | 98.38      | 100.00    
XGBoost            | 1.0000     | 1.0000     | 0.00       | 0.00      
Random Forest      | 1.0000     | 1.0000     | 0.00       | 0.00      
SVM (RBF)          | 1.0000     | 1.0000     | 0.03       | 0.00      
Paper: XGBoost ROC-AUC=0.999, PR-AUC=0.998

  ADDITIONAL METRICS (Dataset B)
Method             | ROC-AUC    | PR-AUC     | FPR (%)    | FNR (%)   
---------------------------------------------------------------------------
Threshold-based    | 0.7743     | 0.8156     | 22.56      | 99.97     
XGBoost            | 1.0000     | 1.0000     | 0.00       | 0.02      
Random Forest      | 1.0000     | 1.0000     | 0.00       | 0.02      
SVM (RBF)          | 1.0000     | 1.0000     | 0.01       | 0.02      
Paper: XGBoost ROC-AUC=0.99

## 8. Statistical Significance (Paired t-test)

In [9]:
def run_ttest(results, dataset_name):
    print(f'\n{"="*60}')
    print(f'  PAIRED T-TEST ({dataset_name}): XGBoost vs Random Forest')
    print(f'{"="*60}')
    
    xgb_f1 = results['XGBoost']['f1']
    rf_f1 = results['Random Forest']['f1']
    
    t_stat, p_value = stats.ttest_rel(xgb_f1, rf_f1)
    
    print(f'  XGBoost F1 per fold: {[f"{v:.4f}" for v in xgb_f1]}')
    print(f'  RF F1 per fold:      {[f"{v:.4f}" for v in rf_f1]}')
    print(f'  t-statistic: {t_stat:.4f}')
    print(f'  p-value: {p_value:.6f}')
    print(f'  df: {len(xgb_f1) - 1}')
    print(f'  Significant (p < 0.01)? {"YES" if p_value < 0.01 else "NO"}')
    print(f'\n  Paper reports: t=4.82, df=4, p<0.01')
    print(f'{"="*60}')

run_ttest(results_a, 'Dataset A')
run_ttest(results_b, 'Dataset B')


  PAIRED T-TEST (Dataset A): XGBoost vs Random Forest
  XGBoost F1 per fold: ['99.9987', '99.9920', '100.0000', '99.9987', '99.9947']
  RF F1 per fold:      ['100.0000', '100.0000', '100.0000', '100.0000', '99.9973']
  t-statistic: -1.9069
  p-value: 0.129200
  df: 4
  Significant (p < 0.01)? NO

  Paper reports: t=4.82, df=4, p<0.01

  PAIRED T-TEST (Dataset B): XGBoost vs Random Forest
  XGBoost F1 per fold: ['99.9960', '99.9853', '99.9933', '99.9947', '99.9867']
  RF F1 per fold:      ['99.9960', '99.9867', '99.9920', '99.9947', '99.9893']
  t-statistic: -0.7845
  p-value: 0.476581
  df: 4
  Significant (p < 0.01)? NO

  Paper reports: t=4.82, df=4, p<0.01


## 9. Kesimpulan

### Pertanyaan yang dijawab:
1. Apakah XGBoost lebih baik dari RF dan SVM pada 10 fitur?
2. Apakah perbedaan signifikan secara statistik?
3. Apakah Threshold-based jauh lebih buruk?
4. Apakah ROC-AUC dan PR-AUC mendekati paper?
5. Apakah FNR rendah (penting untuk security)?